# 01 — Kaggle: Open-LLM Inference

Run this notebook on Kaggle with **GPU enabled** and **Internet enabled**. It performs all LLM inference, saves resumable Parquet checkpoints under `/kaggle/working`, cross-judges Phase 4 with the other open model, and emits `kaggle_results_<profile>.zip`.

In [ ]:
REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
PROFILE = "smoke"  # must match Colab preparation
KAGGLE_INPUT_DIR = "/kaggle/input/PLACEHOLDER-DATASET-SLUG"
REPO_DIR = "/kaggle/working/political-bias-lab"

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"])

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before running inference."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

## Copy read-only Kaggle Dataset inputs into the writable working directory

In [ ]:
from src.cloud import copy_kaggle_input
from pathlib import Path
copy_kaggle_input(KAGGLE_INPUT_DIR, REPO_DIR)
print(sorted(p.name for p in (Path(REPO_DIR)/"prepared").glob("*.parquet")))

In [ ]:
from src.config import load_config
from src.io_utils import read_json
from src.reproducibility import save_run_manifest
from pathlib import Path
import subprocess

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
prep = read_json(Path(REPO_DIR)/"results/manifests/data_preparation.json", {})
if prep:
    assert prep.get("profile") == PROFILE, f"Prepared data profile {prep.get('profile')} does not match Kaggle profile {PROFILE}"

prepare_manifest = read_json(Path(REPO_DIR)/"results/manifests/colab_prepare_manifest.json", {})
current_git = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if prepare_manifest.get("git_revision"):
    assert prepare_manifest["git_revision"] == current_git, (
        "Git revision mismatch. Check out the exact revision used in Colab preparation: ",
        prepare_manifest["git_revision"], current_git
    )

save_run_manifest(Path(REPO_DIR)/"results/manifests/kaggle_inference_manifest.json", config=cfg, root=REPO_DIR, extra={"stage":"kaggle_inference"})
print("Profile:", PROFILE)

## Run evaluated models sequentially

The first model is unloaded before the second model loads, which minimizes VRAM use. Successful rows are checkpointed; failed rows can be retried on a rerun.

In [ ]:
from src.pipeline import run_one_model
from src.io_utils import write_json
from pathlib import Path

model_manifests = []
for model_cfg in cfg["models"]["evaluated"]:
    print("\n=== Running", model_cfg["id"], "===")
    model_manifests.append(run_one_model(cfg, model_cfg, root=REPO_DIR))
write_json(model_manifests, Path(REPO_DIR)/"results/manifests/evaluated_models.json")
model_manifests

## Cross-model Phase-4 judging

Each default evaluated model judges only the other model's outputs, never its own.

In [ ]:
from src.pipeline import run_cross_judging
from src.io_utils import write_json
judge_manifests = run_cross_judging(cfg, root=REPO_DIR)
write_json(judge_manifests, Path(REPO_DIR)/"results/manifests/judge_models.json")
judge_manifests

## Sanity-check failures before export

In [ ]:
import pandas as pd
from pathlib import Path
raw_dir = Path(REPO_DIR)/"results/raw"
for p in sorted(raw_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    failures = int(df["error"].notna().sum()) if "error" in df.columns else 0
    print(p.name, "rows=", len(df), "failures=", failures)

## Export Kaggle output bundle

In [ ]:
from src.transfer import make_kaggle_output_bundle
from pathlib import Path
out = Path("/kaggle/working")/f"kaggle_results_{PROFILE}.zip"
make_kaggle_output_bundle(REPO_DIR, out)
print("Save/commit the notebook, then download this output:", out)